In [1]:
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

from state import TriageState, TriageResult

from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GROQ_MODEL = os.getenv("GROQ_MODEL")


In [3]:
model = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    temperature=0,
)

In [4]:
structured_model = model.with_structured_output(
    TriageResult
)


In [5]:
def classify_with_llm(state: TriageState):
    email = state.get("email", "").strip()

    if not email:
        raise ValueError("Email cannot be empty")

    prompt = f"""
You are an email triage assistant.

Classify the email using the required structured schema.

Classification rules:
- Complaint: A customer reports a problem or dissatisfaction.
- Feedback: A customer provides an opinion or suggestion.
- Request: A customer asks for information or an action.
- Spam: Unwanted, deceptive, or irrelevant messages.
- Other: Does not fit the categories above.

Priority rules:
- Low: No urgency and no significant impact.
- Medium: Requires attention but is not urgent.
- High: Significant issue or time-sensitive request.
- Critical: Severe impact, security concern, or immediate risk.

Additional rules:
- Use only information present in the email.
- Do not invent facts.
- Write a concise summary.
- Set confidence between 0 and 1.
- Consider whether human review is required.

Email:
<email_content>
{email}
</email_content>
"""

    result = structured_model.invoke(prompt)

    return {
        "triage_result": result
    }

In [7]:
def validate_triage(state: TriageState):
    triage_result = state.get("triage_result")

    if triage_result is None:
        return {
            "validation_errors": [
                "Triage result is missing."
            ],
            "review_reason": "Classification result is missing.",
            "route": "human_review",
        }

    errors = []

    # Rule 1: Low-confidence classifications require review.
    if triage_result.confidence < 0.70:
        errors.append(
            "Classification confidence is below 0.70."
        )

    # Rule 2: Critical emails require human review.
    if triage_result.priority == "Critical":
        errors.append(
            "Critical-priority email requires review."
        )

    # Rule 3: High-priority emails require review for now.
    if triage_result.priority == "High":
        errors.append(
            "High-priority email requires review."
        )

    # Rule 4: Respect the model's review recommendation,
    # but do not depend on it exclusively.
    if triage_result.needs_human_review:
        errors.append(
            "Model recommended human review."
        )

    if errors:
        return {
            "validation_errors": errors,
            "review_reason": " ".join(errors),
            "route": "human_review",
        }

    return {
        "validation_errors": [],
        "review_reason": "",
        "route": "automatic_processing",
    }

In [8]:
def route_email(state: TriageState):
    route = state.get("route")

    if route == "human_review":
        return "human_review"

    return "automatic_processing"

In [9]:
def human_review(state: TriageState):
    triage_result = state.get("triage_result")

    print("\n--- HUMAN REVIEW REQUIRED ---")

    if triage_result:
        print("Category:", triage_result.category)
        print("Priority:", triage_result.priority)
        print("Summary:", triage_result.summary)

    print("Reason:", state.get("review_reason", ""))

    return {}

In [10]:
def automatic_processing(state: TriageState):
    triage_result = state.get("triage_result")

    print("\n--- AUTOMATIC PROCESSING ---")

    if triage_result:
        print("Category:", triage_result.category)
        print("Priority:", triage_result.priority)
        print("Summary:", triage_result.summary)

    return {}

In [15]:
builder = StateGraph(TriageState)

# 1. Classification
builder.add_node("classify_with_llm",classify_with_llm)

# 2. Deterministic validation
builder.add_node("validate_triage",validate_triage)

# 3. Destination nodes
builder.add_node("human_review",human_review)
builder.add_node("automatic_processing",automatic_processing)

# 4. Normal edge: classification → validation
builder.add_edge(START,"classify_with_llm")
builder.add_edge("classify_with_llm","validate_triage")

# 5. Conditional edge: validation → selected route
builder.add_conditional_edges("validate_triage",route_email,
    {
        "human_review": "human_review",
        "automatic_processing": "automatic_processing",
    },
)

# 6. Both paths finish the workflow
builder.add_edge("human_review",END)
builder.add_edge("automatic_processing",END)

graph = builder.compile()

